- KG based RAG（Graph RAG）不是 GraphRAG（一个 standalone 的 solution）
- 一个通用的基于 llm 的 kg parse 该如何写
    - general prompt template
- kg parse 时多个 document 解析的内容如何合并；
- app
    - hybrid retrieval：retrieval on neo4j graph
- neo4j integration 相关

### neo4j

- 安装 neo4j-desktop
    - 配置文件：`neo4j.conf`
        - `server.bolt.listen_address=0.0.0.0:7687`
    - http://localhost:7474/browser/  
- 创建实例
    - 插件安装：APOC (插件式实例级别的)
- 参考
    - https://python.langchain.com/docs/how_to/graph_constructing/

- features
    - Neo4j 同时支持关键词和向量索引
- 如果在 store 时设置了 `baseEntityLabel=True`，节点可能会有一个基础标签（例如，基于 `__type__`）。你可以查询特定标签的节点。假设你的文档类型是 "Document"：

```
MATCH (d:Document)
RETURN d
LIMIT 50
```

In [30]:
# !pip install --upgrade --quiet  langchain langchain-neo4j langchain-openai langchain-experimental neo4j

### langchain

In [1]:
# !pip install -U langchain-neo4j

In [2]:
# from langchain_community.graphs import Neo4jGraph
from langchain_neo4j import Neo4jGraph

In [3]:
graph = Neo4jGraph(
    url="neo4j://192.168.194.175:7687",
    username="neo4j",
    password="Apple1105,",
    refresh_schema=False
)
def clean_graph():
    query = """
    MATCH (n)
    DETACH DELETE n
    """
    graph.query(query)

### LLM Graph Transformer

- Graph Transformer: https://python.langchain.com/api_reference/_modules/langchain_experimental/graph_transformers/llm.html#LLMGraphTransformer
    - transformation of unstructured information into structured formats, facilitating deeper insights and more efficient navigation through complex relationships and patterns.
    - The `LLMGraphTransformer` converts text documents into structured graph documents by **leveraging a LLM to parse and categorize entities and their relationships**.
    - The selection of the LLM model significantly influences the output by determining the accuracy and nuance of the extracted graph data.
- parameters
    - allowed_nodes：list
    - allowed_relationships: list
        - single: `NATIONALITY`
        - three-tuple: `"Person", "NATIONALITY", "Country"`
    - node_properties
        - When set to True, LLM autonomously identifies and extracts relevant node properties.
        - Conversely, if node_properties is defined as a list of strings, the LLM selectively retrieves only the specified properties from the text.
    - ignore_tool_usage：默认为 false，即会把“图抽取”的输出 Schema（节点、关系及其属性的结构）作为一个“工具”绑定到模型，让模型按这个 Schema 返回结构化结果；
- neo4j_graph.add_graph_documents
    - baseEntityLabel
        - Most graph databases support indexes to optimize data import and retrieval.
    - include_source
        - This approach lets us track which documents each entity appeared in.

In [4]:
from langchain_core.documents import Document

text = """
Marie Curie, 7 November 1867 – 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.
She was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.
Her husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.
She was, in 1906, the first woman to become a professor at the University of Paris.
Also, Robin Williams!
"""
documents = [Document(page_content=text)]

In [10]:
from dotenv import load_dotenv, find_dotenv
assert load_dotenv(find_dotenv())

In [11]:
from langchain.chat_models import init_chat_model
llm = init_chat_model(model='gpt-4.1-mini', model_provider='openai')

In [12]:
from langchain_experimental.graph_transformers import LLMGraphTransformer
no_schema = LLMGraphTransformer(llm=llm)

In [13]:
data = await no_schema.aconvert_to_graph_documents(documents)

In [8]:
# data

In [14]:
from rich.pretty import pprint
pprint(data)

[
│   GraphDocument(
│   │   nodes=[
│   │   │   Node(id='Marie Curie', type='Person', properties={}),
│   │   │   Node(id='Pierre Curie', type='Person', properties={}),
│   │   │   Node(id='Nobel Prize', type='Award', properties={}),
│   │   │   Node(id='University Of Paris', type='Organization', properties={}),
│   │   │   Node(id='Radioactivity', type='Concept', properties={})
│   │   ],
│   │   relationships=[
│   │   │   Relationship(
│   │   │   │   source=Node(id='Marie Curie', type='Person', properties={}),
│   │   │   │   target=Node(id='Radioactivity', type='Concept', properties={}),
│   │   │   │   type='RESEARCH',
│   │   │   │   properties={}
│   │   │   ),
│   │   │   Relationship(
│   │   │   │   source=Node(id='Marie Curie', type='Person', properties={}),
│   │   │   │   target=Node(id='Nobel Prize', type='Award', properties={}),
│   │   │   │   type='AWARDED',
│   │   │   │   properties={}
│   │   │   ),
│   │   │   Relationship(
│   │   │   │   source=Node(id='Pierre Curie', type='Person', properties={}),
│   │   │   │   target=Node(id='Nobel Prize', type='Award', properties={}),
│   │   │   │   type='AWARDED',
│   │   │   │   properties={}
│   │   │   ),
│   │   │   Relationship(
│   │   │   │   source=Node(id='Marie Curie', type='Person', properties={}),
│   │   │   │   target=Node(id='Pierre Curie', type='Person', properties={}),
│   │   │   │   type='SPOUSE',
│   │   │   │   properties={}
│   │   │   ),
│   │   │   Relationship(
│   │   │   │   source=Node(id='Marie Curie', type='Person', properties={}),
│   │   │   │   target=Node(id='University Of Paris', type='Organization', properties={}),
│   │   │   │   type='PROFESSOR',
│   │   │   │   properties={}
│   │   │   )
│   │   ],
│   │   source=Document(
│   │   │   metadata={},
│   │   │   page_content='\nMarie Curie, 7 November 1867 – 4 July 1934, was a Polish and naturalised-French physicist and chemist who conducted pioneering research on radioactivity.\nShe was the first woman to win a Nobel Prize, the first person to win a Nobel Prize twice, and the only person to win a Nobel Prize in two scientific fields.\nHer husband, Pierre Curie, was a co-winner of her first Nobel Prize, making them the first-ever married couple to win the Nobel Prize and launching the Curie family legacy of five Nobel Prizes.\nShe was, in 1906, the first woman to become a professor at the University of Paris.\nAlso, Robin Williams!\n'
│   │   )
│   )
]

In [15]:
graph.add_graph_documents(data)

In [17]:
clean_graph()

In [18]:
no_schema_prompt = LLMGraphTransformer(llm=llm, ignore_tool_usage=True)
data = await no_schema_prompt.aconvert_to_graph_documents(documents)
graph.add_graph_documents(data)

In [19]:
clean_graph()

### Defining allowed nodes 

In [29]:
allowed_nodes = ["Person", "Organization", "Location", "Award", "ResearchField"]
nodes_defined = LLMGraphTransformer(llm=llm, allowed_nodes=allowed_nodes)
data = await nodes_defined.aconvert_to_graph_documents(documents,)
# graph.add_graph_documents(data, include_source=True)
graph.add_graph_documents(data, baseEntityLabel=True)

In [28]:
clean_graph()

### Defining allowed relationships

In [16]:
allowed_nodes = ["Person", "Organization", "Place", "Award", "ResearchField"]
allowed_relationships = ["SPOUSE", "AWARD", "FIELD_OF_RESEARCH", "WORKS_AT", "IN_LOCATION"]
rels_defined = LLMGraphTransformer(
  llm=llm,
  allowed_nodes=allowed_nodes,
  allowed_relationships=allowed_relationships
)
data = await rels_defined.aconvert_to_graph_documents(documents)
graph.add_graph_documents(data)

In [17]:
clean_graph()

In [18]:
allowed_nodes = ["Person", "Organization", "Location", "Award", "ResearchField"]
allowed_relationships = [
    ("Person", "SPOUSE", "Person"),
    ("Person", "AWARD", "Award"),
    ("Person", "WORKS_AT", "Organization"),
    ("Organization", "IN_LOCATION", "Location"),
    ("Person", "FIELD_OF_RESEARCH", "ResearchField")
]
rels_defined = LLMGraphTransformer(
  llm=llm,
  allowed_nodes=allowed_nodes,
  allowed_relationships=allowed_relationships
)
data = await rels_defined.aconvert_to_graph_documents(documents)

### GraphTransformer（源码 or prompt学习）

- 一个通用的基于 llm 的 kg parse 以及实例化该如何写
- kg parse 时多个 document 解析的内容如何合并；
    - 该实现底层用 MERGE / apoc.merge.* 按 id（以及关系三元组）去重合并节点与边，所以插入时就会自动合并重复实体/关系，不需要你手工聚合。
    ```python
    # Neo4jGraph
    graph.add_graph_documents(
        graph_documents,
        baseEntityLabel=True,
        include_source=True
    )
    ```

In [31]:
system_prompt = (
    "# Knowledge Graph Instructions for GPT-4\n"
    "## 1. Overview\n"
    "You are a top-tier algorithm designed for extracting information in structured "
    "formats to build a knowledge graph.\n"
    "Try to capture as much information from the text as possible without "
    "sacrificing accuracy. Do not add any information that is not explicitly "
    "mentioned in the text.\n"
    "- **Nodes** represent entities and concepts.\n"
    "- The aim is to achieve simplicity and clarity in the knowledge graph, making it\n"
    "accessible for a vast audience.\n"
    "## 2. Labeling Nodes\n"
    "- **Consistency**: Ensure you use available types for node labels.\n"
    "Ensure you use basic or elementary types for node labels.\n"
    "- For example, when you identify an entity representing a person, "
    "always label it as **'person'**. Avoid using more specific terms "
    "like 'mathematician' or 'scientist'."
    "- **Node IDs**: Never utilize integers as node IDs. Node IDs should be "
    "names or human-readable identifiers found in the text.\n"
    "- **Relationships** represent connections between entities or concepts.\n"
    "Ensure consistency and generality in relationship types when constructing "
    "knowledge graphs. Instead of using specific and momentary types "
    "such as 'BECAME_PROFESSOR', use more general and timeless relationship types "
    "like 'PROFESSOR'. Make sure to use general and timeless relationship types!\n"
    "## 3. Coreference Resolution\n"
    "- **Maintain Entity Consistency**: When extracting entities, it's vital to "
    "ensure consistency.\n"
    'If an entity, such as "John Doe", is mentioned multiple times in the text '
    'but is referred to by different names or pronouns (e.g., "Joe", "he"),'
    "always use the most complete identifier for that entity throughout the "
    'knowledge graph. In this example, use "John Doe" as the entity ID.\n'
    "Remember, the knowledge graph should be coherent and easily understandable, "
    "so maintaining consistency in entity references is crucial.\n"
    "## 4. Strict Compliance\n"
    "Adhere to the rules strictly. Non-compliance will result in termination."
)


In [32]:
print(system_prompt)

# Knowledge Graph Instructions for GPT-4
## 1. Overview
You are a top-tier algorithm designed for extracting information in structured formats to build a knowledge graph.
Try to capture as much information from the text as possible without sacrificing accuracy. Do not add any information that is not explicitly mentioned in the text.
- **Nodes** represent entities and concepts.
- The aim is to achieve simplicity and clarity in the knowledge graph, making it
accessible for a vast audience.
## 2. Labeling Nodes
- **Consistency**: Ensure you use available types for node labels.
Ensure you use basic or elementary types for node labels.
- For example, when you identify an entity representing a person, always label it as **'person'**. Avoid using more specific terms like 'mathematician' or 'scientist'.- **Node IDs**: Never utilize integers as node IDs. Node IDs should be names or human-readable identifiers found in the text.
- **Relationships** represent connections between entities or concep